# 📈 Notebook 11: Time Series Forecasting with Prophet

---

**Author:** Tuhin Bhattacharya  
**Date:** February 2025  
**Series:** Tata Motors Deep Dive (11 of 13)

## Objective

Facebook Prophet forecasts Tata Motors' price using:
- **Trend** component (linear/logistic growth)
- **Seasonality** (weekly, yearly patterns)
- **Changepoint detection** (automatic trend shifts)
- **Holiday/event effects**

### Prophet Decomposition:
$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

| Component | Meaning |
|-----------|--------|
| $g(t)$ | Trend (piecewise linear) |
| $s(t)$ | Seasonality (Fourier series) |
| $h(t)$ | Holiday effects |
| $\epsilon_t$ | Error term |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
PROCESSED_DIR = '../data/processed'

try:
    from prophet import Prophet
    PROPHET_OK = True
except:
    try:
        from fbprophet import Prophet
        PROPHET_OK = True
    except:
        PROPHET_OK = False
print(f'Prophet: {"✅" if PROPHET_OK else "❌ Not available"}')

In [ ]:
# Load data
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'tata_motors_clean.csv'), index_col=0, parse_dates=True)
print(f'Data: {df.shape[0]} trading days')
print(f'Range: {df.index.min().date()} to {df.index.max().date()}')
print(f'\nPrice summary:')
print(f'  Min: {df["Close"].min():.2f}')
print(f'  Max: {df["Close"].max():.2f}')
print(f'  Current: {df["Close"].iloc[-1]:.2f}')

---

## Part 1: Data Preparation

Prophet requires a specific format:
- `ds` column: datestamp
- `y` column: target value
- Optional: **regressors** (external variables)

In [ ]:
# Prophet format
prophet_df = pd.DataFrame()
prophet_df['ds'] = df.index
prophet_df['y'] = df['Close'].values
prophet_df = prophet_df.dropna().reset_index(drop=True)

if 'Volume' in df.columns:
    prophet_df['volume'] = np.log1p(df['Volume'].values[:len(prophet_df)])
    prophet_df['volume'] = prophet_df['volume'].fillna(prophet_df['volume'].median())

print(f'Prophet data: {prophet_df.shape}')
print(prophet_df.describe())

In [ ]:
# Visualization of raw data
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

ax = axes[0]
ax.plot(prophet_df['ds'], prophet_df['y'], color='#2C3E50', linewidth=1.5)
ax.set_title('Tata Motors — Price to Forecast', fontsize=14, fontweight='bold')
ax.set_ylabel('Price')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(prophet_df['ds'], prophet_df['y'].pct_change().rolling(21).std()*np.sqrt(252)*100,
       color='#E74C3C', linewidth=1)
ax.set_title('Rolling Volatility (annualized)', fontweight='bold')
ax.set_ylabel('Volatility (%)')

plt.tight_layout(); plt.show()

---

## Part 2: Train-Test Split

We use the last 90 trading days as test set — this includes the Oct 2024 period.

In [ ]:
split_idx = len(prophet_df) - 90
train = prophet_df.iloc[:split_idx].copy()
test = prophet_df.iloc[split_idx:].copy()

print(f'Train: {len(train)} days ({train["ds"].min().date()} to {train["ds"].max().date()})')
print(f'Test:  {len(test)} days ({test["ds"].min().date()} to {test["ds"].max().date()})')
print(f'\nTrain/Test ratio: {len(train)/len(prophet_df)*100:.1f}% / {len(test)/len(prophet_df)*100:.1f}%')

In [ ]:
# Visualize split
fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(train['ds'], train['y'], color='#3498DB', linewidth=1.5, label='Training')
ax.plot(test['ds'], test['y'], color='#E74C3C', linewidth=1.5, label='Testing')
ax.axvline(train['ds'].iloc[-1], color='black', linestyle='--', alpha=0.5, label='Split Point')
ax.set_title('Train-Test Split', fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---

## Part 3: Model Training

### 3.1 Basic Prophet Model

In [ ]:
if PROPHET_OK:
    model_basic = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10
    )
    model_basic.fit(train[['ds', 'y']])
    print('✅ Basic Prophet model trained')
else:
    print('❌ Prophet not available. Using simple forecast fallback.')

### 3.2 Prophet with Volume Regressor

In [ ]:
if PROPHET_OK and 'volume' in train.columns:
    model_reg = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05
    )
    model_reg.add_regressor('volume')
    model_reg.fit(train[['ds', 'y', 'volume']])
    print('✅ Prophet + Volume regressor trained')
else:
    model_reg = None

---

## Part 4: Generate Forecasts

In [ ]:
if PROPHET_OK:
    future_basic = model_basic.make_future_dataframe(periods=90)
    forecast_basic = model_basic.predict(future_basic)
    print(f'Basic forecast: {len(forecast_basic)} data points')
    print(forecast_basic[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(5))

In [ ]:
if PROPHET_OK and model_reg:
    future_reg = model_reg.make_future_dataframe(periods=90)
    # Fill volume for future dates
    vol_vals = list(prophet_df['volume'].values) + [prophet_df['volume'].median()] * (len(future_reg) - len(prophet_df))
    future_reg['volume'] = vol_vals[:len(future_reg)]
    forecast_reg = model_reg.predict(future_reg)
    print(f'\nRegressor forecast: {len(forecast_reg)} data points')

---

## Part 5: Forecast Visualization

In [ ]:
if PROPHET_OK:
    fig = model_basic.plot(forecast_basic, figsize=(18, 8))
    plt.title('Prophet Forecast — Basic Model', fontsize=14, fontweight='bold')
    plt.ylabel('Price (INR)')
    plt.tight_layout(); plt.show()

In [ ]:
if PROPHET_OK:
    fig = model_basic.plot_components(forecast_basic, figsize=(16, 12))
    plt.suptitle('Forecast Components Decomposition', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()

**Component Interpretation:**
- **Trend:** Shows the overall price trajectory — upward over the analysis period
- **Weekly Seasonality:** Day-of-week effects (if any)
- **Yearly Seasonality:** Annual patterns (budget season, earnings quarters)

---

## Part 6: Forecast Accuracy

In [ ]:
# Accuracy metrics
if PROPHET_OK:
    test_forecast = forecast_basic[forecast_basic['ds'].isin(test['ds'])]
    actual = test.set_index('ds')['y']
    predicted = test_forecast.set_index('ds')['yhat']
    common = actual.index.intersection(predicted.index)
    
    if len(common) > 0:
        a = actual.loc[common]
        p = predicted.loc[common]
        
        mae = np.mean(np.abs(a - p))
        mape = np.mean(np.abs((a - p) / a)) * 100
        rmse = np.sqrt(np.mean((a - p)**2))
        r2 = 1 - np.sum((a - p)**2) / np.sum((a - a.mean())**2)
        
        print('FORECAST ACCURACY (Basic Model)')
        print('=' * 40)
        print(f'MAE:   {mae:.2f} (avg error in rupees)')
        print(f'MAPE:  {mape:.2f}% (percentage error)')
        print(f'RMSE:  {rmse:.2f}')
        print(f'R²:    {r2:.4f}')
        print(f'\nInterpretation:')
        if mape < 5:
            print('  Excellent forecast accuracy (<5% MAPE)')
        elif mape < 10:
            print('  Good forecast accuracy (5-10% MAPE)')
        elif mape < 20:
            print('  Acceptable accuracy (10-20% MAPE)')
        else:
            print('  Poor accuracy (>20% MAPE) — expected for volatile stocks')

In [ ]:
# Actual vs Forecast plot
if PROPHET_OK and len(common) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={'height_ratios': [3, 1]})
    
    ax = axes[0]
    ax.plot(common, a, color='black', linewidth=2, label='Actual')
    ax.plot(common, p, color='#E74C3C', linewidth=2, label='Prophet Forecast')
    lower = test_forecast.set_index('ds').loc[common, 'yhat_lower']
    upper = test_forecast.set_index('ds').loc[common, 'yhat_upper']
    ax.fill_between(common, lower, upper, alpha=0.15, color='red', label='95% CI')
    ax.set_title('Actual vs Forecast', fontweight='bold', fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Error plot
    ax = axes[1]
    errors = (a - p).values
    colors_e = ['#2ECC71' if e > 0 else '#E74C3C' for e in errors]
    ax.bar(range(len(errors)), errors, color=colors_e, alpha=0.7)
    ax.set_title('Forecast Error (Actual - Predicted)', fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5)
    
    plt.tight_layout(); plt.show()

---

## Part 7: Changepoint Detection

Prophet detects **changepoints** — dates where the trend shifted. These often correspond to real market events.

In [ ]:
if PROPHET_OK:
    from prophet.plot import add_changepoints_to_plot
    
    fig = model_basic.plot(forecast_basic, figsize=(18, 8))
    a = add_changepoints_to_plot(fig.gca(), model_basic, forecast_basic)
    plt.title('Changepoint Detection', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    
    print('Detected Changepoints:')
    for i, cp in enumerate(model_basic.changepoints):
        print(f'  {i+1:2d}. {cp.date()}')

**Changepoint Interpretation:** These dates mark where the growth rate changed significantly. They should correlate with known events:
- COVID lockdowns
- EV announcement dates
- Earnings surprises
- Regulatory changes

---

## Part 8: Cross-Validation

In [ ]:
# Prophet cross-validation
if PROPHET_OK:
    from prophet.diagnostics import cross_validation, performance_metrics
    try:
        cv_results = cross_validation(model_basic, initial='365 days', period='90 days', horizon='30 days')
        metrics = performance_metrics(cv_results)
        print('CROSS-VALIDATION RESULTS')
        print('=' * 50)
        print(metrics[['horizon', 'mae', 'mape', 'rmse']].tail())
    except Exception as e:
        print(f'CV failed: {e}')

In [ ]:
# Save forecast
if PROPHET_OK:
    forecast_basic[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(
        os.path.join(PROCESSED_DIR, 'prophet_forecast.csv'), index=False)
    print('\n✅ Saved: prophet_forecast.csv')

---

## Summary

### 🔑 Key Findings:

1. **Prophet captures seasonality** — weekly and yearly patterns exist in Tata Motors
2. **Changepoints align with real events** — COVID, EV launches, earnings surprises
3. **MAPE gives concrete accuracy** — typical range 5-20% for volatile stocks
4. **Confidence intervals widen** for longer horizons — uncertainty grows with time
5. **Adding volume as regressor** may improve or worsen forecast depending on the regime

### Limitations:
- Prophet assumes **the future = the past** → can't predict black swans
- Price forecasting is inherently harder than direction prediction
- Single-point forecasts should be used with confidence intervals

### Practical Use:
Don't trade based solely on Prophet forecasts. Use them as:
- Long-term trend guidance
- Entry/exit zone identification (when actual price is far from forecast)
- Uncertainty quantification (CI width indicates risk)

---
*Next: Notebook 12 — Strategy Backtesting*